# 02 - Simulador, feedback y logs

Este notebook muestra el flujo completo sin servicios externos: cierre, corrector identity, feedback JSONL y logs de debugging.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "realtime").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

## Correr el flujo en un directorio temporal

Usamos `tempfile` para no dejar artefactos versionables. En uso normal, los outputs van a `realtime/outputs/`, que esta ignorado por git.

In [ ]:
import json
import tempfile
from pathlib import Path

from realtime.src.simular_flujo import DEMO_TEXTS, run_simulation

tmp = tempfile.TemporaryDirectory()
base = Path(tmp.name)
summary = run_simulation(
    DEMO_TEXTS,
    log_path=base / "logs.jsonl",
    feedback_path=base / "feedback.jsonl",
)
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Ver un evento de feedback

Solo se escribe feedback cuando hubo `commit`. La decision de usuario queda como `unclear` hasta que alguien revise o corrija.

In [ ]:
feedback_rows = (base / "feedback.jsonl").read_text(encoding="utf-8").strip().splitlines()
print("eventos_feedback", len(feedback_rows))
print(json.dumps(json.loads(feedback_rows[0]), indent=2, ensure_ascii=False))

## Ver un log de provider

Cada log guarda provider, etapa, input, output, latencia, validacion y si hubo fallback. Esto permite debuggear sin depender de notebooks.

In [ ]:
log_rows = (base / "logs.jsonl").read_text(encoding="utf-8").strip().splitlines()
print("eventos_log", len(log_rows))
print(json.dumps(json.loads(log_rows[0]), indent=2, ensure_ascii=False))

## Provider LLM opcional

Cuando haya Ollama local, se puede probar sin cambiar el resto del flujo:

```python
run_simulation(DEMO_TEXTS, closure_provider_name="ollama", ollama_model="qwen3:4b")
```

Si Ollama no esta disponible, el provider cae a `wait` y registra fallback.